In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:")
print(PROJECT_ROOT)

Project root:
C:\Users\User\RAG SYS\Question_Generation


In [2]:
from rag.hybrid_retriever import HybridRetriever

hybrid = HybridRetriever()

INITIALIZING HYBRID RETRIEVER

Loading dense retriever...
Loading embedding model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding model loaded.
Embedding dimension: 384
Loading FAISS index...
Loading metadata...

RETRIEVER INITIALIZED
Vectors: 2150
Metadata: 2150
Dimension: 384
Top-K: 20
Dense retriever ready.

Loading BM25 retriever...
INITIALIZING BM25

Loading question data...
Records loaded: 2150
Questions extracted: 2150

Building BM25 index...
BM25 index built successfully.

BM25 READY
BM25 retriever ready.

HYBRID RETRIEVER READY
Dense top-k : 20
BM25 top-k  : 20
RRF k       : 60


In [3]:
query = "Explain object detection in computer vision."

results = hybrid.retrieve(
    query,
    top_k=10
)

print("=" * 80)
print("HYBRID RETRIEVAL")
print("=" * 80)

print("QUERY:", query)

print("\nRESULTS:")

for rank, result in enumerate(results, 1):

    metadata = result["metadata"]

    # Your metadata is nested, so unwrap it
    if "metadata" in metadata:
        metadata = metadata["metadata"]

    print(
        f"\n{rank}. "
        f"[RRF: {result['rrf_score']:.6f}] "
        f"{result['id']}"
    )

    print(
        f"   Dense score: "
        f"{result['dense_score']:.4f}"
    )

    print(
        f"   BM25 score: "
        f"{result['bm25_score']:.4f}"
    )

    print(
        f"   Dense rank: "
        f"{result['dense_rank']}"
    )

    print(
        f"   BM25 rank: "
        f"{result['bm25_rank']}"
    )

    print(
        f"   Question: "
        f"{metadata['question']}"
    )

HYBRID RETRIEVAL
QUERY: Explain object detection in computer vision.

RESULTS:

1. [RRF: 0.031778] ai_ml_computer_vision_q021
   Dense score: 0.7911
   BM25 score: 6.8252
   Dense rank: 1
   BM25 rank: 5
   Question: What is object detection?

2. [RRF: 0.031010] ai_ml_computer_vision_q023
   Dense score: 0.7384
   BM25 score: 7.0340
   Dense rank: 5
   BM25 rank: 4
   Question: What is a bounding box in object detection?

3. [RRF: 0.029199] ai_ml_computer_vision_q042
   Dense score: 0.7234
   BM25 score: 6.2314
   Dense rank: 8
   BM25 rank: 9
   Question: What is the difference between face detection and face recognition?

4. [RRF: 0.028787] ai_ml_computer_vision_q022
   Dense score: 0.7547
   BM25 score: 4.9165
   Dense rank: 2
   BM25 rank: 19
   Question: What is the difference between image classification and object detection?

5. [RRF: 0.028382] ai_ml_computer_vision_q002
   Dense score: 0.7225
   BM25 score: 5.8031
   Dense rank: 9
   BM25 rank: 12
   Question: What are common a

In [4]:
# ============================================================
# HYBRID RETRIEVAL — PARAPHRASE EVALUATION
# ============================================================

import json
from pathlib import Path

EVAL_FILE = Path(
    r"C:\Users\User\RAG SYS\Question_Generation\tests\retrieval_eval.json"
)

with open(EVAL_FILE, "r", encoding="utf-8") as f:
    evaluation_data = json.load(f)

print("=" * 70)
print("HYBRID RETRIEVAL EVALUATION")
print("=" * 70)

print(f"\nEvaluation queries: {len(evaluation_data)}")


# ============================================================
# METRICS
# ============================================================

recall_at_1 = 0
recall_at_3 = 0
recall_at_5 = 0

reciprocal_ranks = []

total = len(evaluation_data)


# ============================================================
# EVALUATION
# ============================================================

for i, item in enumerate(evaluation_data, 1):

    query = item["query"]
    expected_id = item["expected_id"]

    results = hybrid.retrieve(
        query,
        top_k=5
    )

    retrieved_ids = [
        result["id"]
        for result in results
    ]

    # --------------------------------------------------------
    # Recall@1
    # --------------------------------------------------------

    if expected_id in retrieved_ids[:1]:
        recall_at_1 += 1

    # --------------------------------------------------------
    # Recall@3
    # --------------------------------------------------------

    if expected_id in retrieved_ids[:3]:
        recall_at_3 += 1

    # --------------------------------------------------------
    # Recall@5
    # --------------------------------------------------------

    if expected_id in retrieved_ids[:5]:
        recall_at_5 += 1

    # --------------------------------------------------------
    # MRR
    # --------------------------------------------------------

    if expected_id in retrieved_ids:

        rank = retrieved_ids.index(expected_id) + 1

        reciprocal_ranks.append(1 / rank)

    else:

        reciprocal_ranks.append(0)

    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    if i % 20 == 0 or i == total:

        print(
            f"Processed {i}/{total} "
            f"({i / total * 100:.1f}%)"
        )


# ============================================================
# FINAL METRICS
# ============================================================

recall_1 = recall_at_1 / total
recall_3 = recall_at_3 / total
recall_5 = recall_at_5 / total

mrr = sum(reciprocal_ranks) / total


# ============================================================
# RESULTS
# ============================================================

print("\n")
print("=" * 70)
print("HYBRID FINAL RESULTS")
print("=" * 70)

print(f"\nTotal queries: {total}")

print(f"Recall@1: {recall_1 * 100:.2f}%")
print(f"Recall@3: {recall_3 * 100:.2f}%")
print(f"Recall@5: {recall_5 * 100:.2f}%")
print(f"MRR:      {mrr:.4f}")

print("\n" + "=" * 70)

HYBRID RETRIEVAL EVALUATION

Evaluation queries: 200
Processed 20/200 (10.0%)
Processed 40/200 (20.0%)
Processed 60/200 (30.0%)
Processed 80/200 (40.0%)
Processed 100/200 (50.0%)
Processed 120/200 (60.0%)
Processed 140/200 (70.0%)
Processed 160/200 (80.0%)
Processed 180/200 (90.0%)
Processed 200/200 (100.0%)


HYBRID FINAL RESULTS

Total queries: 200
Recall@1: 96.50%
Recall@3: 100.00%
Recall@5: 100.00%
MRR:      0.9825



In [6]:
# ============================================================
# INITIALIZE HYBRID RETRIEVER
# ============================================================

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:")
print(PROJECT_ROOT)

# Import hybrid retriever
from rag.hybrid_retriever import HybridRetriever

print("\nInitializing Hybrid Retriever...")

hybrid = HybridRetriever()

print("\nHybrid Retriever ready!")

Project root:
C:\Users\User\RAG SYS\Question_Generation

Initializing Hybrid Retriever...
INITIALIZING HYBRID RETRIEVER

Loading dense retriever...
Loading embedding model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding model loaded.
Embedding dimension: 384
Loading FAISS index...
Loading metadata...

RETRIEVER INITIALIZED
Vectors: 2150
Metadata: 2150
Dimension: 384
Top-K: 20
Dense retriever ready.

Loading BM25 retriever...
INITIALIZING BM25

Loading question data...
Records loaded: 2150
Questions extracted: 2150

Building BM25 index...
BM25 index built successfully.

BM25 READY
BM25 retriever ready.

HYBRID RETRIEVER READY
Dense top-k : 20
BM25 top-k  : 20
RRF k       : 60

Hybrid Retriever ready!


In [9]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:")
print(PROJECT_ROOT)

from rag.hybrid_retriever import HybridRetriever

print("\nInitializing Hybrid Retriever...")

hybrid = HybridRetriever()

print("\nHybrid Retriever ready!")

Project root:
C:\Users\User\RAG SYS\Question_Generation

Initializing Hybrid Retriever...
INITIALIZING HYBRID RETRIEVER

Loading dense retriever...
Loading embedding model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding model loaded.
Embedding dimension: 384
Loading FAISS index...
Loading metadata...

RETRIEVER INITIALIZED
Vectors: 2150
Metadata: 2150
Dimension: 384
Top-K: 20
Dense retriever ready.

Loading BM25 retriever...
INITIALIZING BM25

Loading question data...
Records loaded: 2150
Questions extracted: 2150

Building BM25 index...
BM25 index built successfully.

BM25 READY
BM25 retriever ready.

HYBRID RETRIEVER READY
Dense top-k : 20
BM25 top-k  : 20
RRF k       : 60

Hybrid Retriever ready!


In [13]:
print("Dense retriever:")
print(hybrid.dense_retriever.__dict__.keys())

print("\nBM25 retriever:")
print(hybrid.bm25_retriever.__dict__.keys())

Dense retriever:
dict_keys(['project_root', 'embeddings_file', 'index_file', 'metadata_file', 'top_k', 'model', 'index', 'metadata', 'embedded_questions'])

BM25 retriever:
dict_keys(['questions', 'metadata', 'bm25'])


In [14]:
# ============================================================
# HYBRID — FULL 2,150 QUESTION EVALUATION
# ============================================================

print("=" * 70)
print("HYBRID FULL RETRIEVAL EVALUATION")
print("=" * 70)

# ------------------------------------------------------------
# Get all 2,150 questions from BM25 component
# ------------------------------------------------------------

questions = hybrid.bm25_retriever.questions
metadata = hybrid.bm25_retriever.metadata

total = len(questions)

print(f"\nTotal questions: {total}")


# ============================================================
# METRICS
# ============================================================

recall_at_1 = 0
recall_at_3 = 0
recall_at_5 = 0

reciprocal_ranks = []

failed = []


# ============================================================
# EVALUATE EVERY QUESTION
# ============================================================

for i, question in enumerate(questions):

    # --------------------------------------------------------
    # Get expected ID
    # --------------------------------------------------------

    meta = metadata[i]

    # Your metadata can contain nested metadata,
    # so handle both structures safely.

    if "id" in meta:
        expected_id = meta["id"]

    elif "metadata" in meta and "id" in meta["metadata"]:
        expected_id = meta["metadata"]["id"]

    else:
        raise KeyError(
            f"Cannot find ID in metadata at index {i}: {meta}"
        )

    # --------------------------------------------------------
    # Hybrid retrieval
    # --------------------------------------------------------

    results = hybrid.retrieve(
        question,
        top_k=5
    )

    retrieved_ids = [
        result["id"]
        for result in results
    ]

    # --------------------------------------------------------
    # Recall@1
    # --------------------------------------------------------

    if expected_id in retrieved_ids[:1]:
        recall_at_1 += 1

    # --------------------------------------------------------
    # Recall@3
    # --------------------------------------------------------

    if expected_id in retrieved_ids[:3]:
        recall_at_3 += 1

    # --------------------------------------------------------
    # Recall@5
    # --------------------------------------------------------

    if expected_id in retrieved_ids[:5]:
        recall_at_5 += 1

    # --------------------------------------------------------
    # MRR
    # --------------------------------------------------------

    if expected_id in retrieved_ids:

        rank = retrieved_ids.index(expected_id) + 1

        reciprocal_ranks.append(1 / rank)

    else:

        reciprocal_ranks.append(0)

        failed.append({
            "question_number": i + 1,
            "query": question,
            "expected_id": expected_id,
            "retrieved_ids": retrieved_ids
        })

    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    if (i + 1) % 100 == 0 or (i + 1) == total:

        print(
            f"Processed {i + 1}/{total} "
            f"({(i + 1) / total * 100:.1f}%)"
        )


# ============================================================
# CALCULATE FINAL METRICS
# ============================================================

recall_1 = recall_at_1 / total
recall_3 = recall_at_3 / total
recall_5 = recall_at_5 / total

mrr = sum(reciprocal_ranks) / total


# ============================================================
# FINAL RESULTS
# ============================================================

print("\n")
print("=" * 70)
print("HYBRID FULL RESULTS")
print("=" * 70)

print(f"\nTotal questions: {total}")

print(f"Recall@1: {recall_1 * 100:.2f}%")
print(f"Recall@3: {recall_3 * 100:.2f}%")
print(f"Recall@5: {recall_5 * 100:.2f}%")
print(f"MRR:      {mrr:.4f}")

print(f"\nFailed @5: {len(failed)}")

print("\n" + "=" * 70)

HYBRID FULL RETRIEVAL EVALUATION

Total questions: 2150
Processed 100/2150 (4.7%)
Processed 200/2150 (9.3%)
Processed 300/2150 (14.0%)
Processed 400/2150 (18.6%)
Processed 500/2150 (23.3%)
Processed 600/2150 (27.9%)
Processed 700/2150 (32.6%)
Processed 800/2150 (37.2%)
Processed 900/2150 (41.9%)
Processed 1000/2150 (46.5%)
Processed 1100/2150 (51.2%)
Processed 1200/2150 (55.8%)
Processed 1300/2150 (60.5%)
Processed 1400/2150 (65.1%)
Processed 1500/2150 (69.8%)
Processed 1600/2150 (74.4%)
Processed 1700/2150 (79.1%)
Processed 1800/2150 (83.7%)
Processed 1900/2150 (88.4%)
Processed 2000/2150 (93.0%)
Processed 2100/2150 (97.7%)
Processed 2150/2150 (100.0%)


HYBRID FULL RESULTS

Total questions: 2150
Recall@1: 97.35%
Recall@3: 100.00%
Recall@5: 100.00%
MRR:      0.9866

Failed @5: 0

